# Phase 4 · CJPE · data, length statistics and a no-language-model baseline
Run this first (CPU runtime is enough). It tells you the case lengths in tokens (how many 512-token chunks a case really has), the label balance, and what a TF-IDF + logistic-regression model scores on the dev set using the **whole case**, the **first third**, the **middle third** and the **last third**. The language-model runs have to beat that to show the Phase 1 model adds anything, and the thirds are a cheap first answer to 'which part of the judgment carries the signal?'.

In [ ]:
# 1 · Settings
from pathlib import Path
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_4_CJPE'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'
RR_LIB, CJPE_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py'
SEED, CHUNK_LEN, MAX_TOKENS = 17, 512, 40000

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data (downloads Phase_4_CJPE from the gated HF dataset if not already on Drive) + tokenisation
from google.colab import userdata
try: token = userdata.get('HF_TOKEN')
except Exception: token = None
J.ensure_files(COURSE_ROOT, token)
DATA = J.load_all(PHASE_DIR)
for s in DATA.values(): J.tokenize_docs(s, TOK, MAX_TOKENS)
print(json.dumps(J.summarize(DATA), indent=1))
ids = [d['id'] for d in DATA['test']]
assert len(ids) == len(set(ids)), 'duplicate test ids'
assert all(d['label'] in (0, 1) for d in DATA['train'] + DATA['dev']), 'labels must be 0/1'
print('chunks of CHUNK_LEN per train case: mean', round(np.mean([len(J.chunk_spans(len(d['tok']), CHUNK_LEN, CHUNK_LEN)) for d in DATA['train']]),1), '| p90', int(np.percentile([len(J.chunk_spans(len(d['tok']), CHUNK_LEN, CHUNK_LEN)) for d in DATA['train']], 90)))

In [ ]:
# 4 · TF-IDF + logistic regression on the whole case and on each third (no language model, CPU)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import numpy as np, pandas as pd
def part(text, which):
    n = len(text); a, b = {'whole': (0, 1), 'first third': (0, 1/3), 'middle third': (1/3, 2/3), 'last third': (2/3, 1)}[which]
    return text[int(a*n):int(b*n)]
TR = [DATA['train'][i] for i in np.random.RandomState(0).permutation(len(DATA['train']))[:8000]]   # subsample: TF-IDF over all ~31k full cases is slow on CPU
ytr = np.array([d['label'] for d in TR]); ydv = np.array([d['label'] for d in DATA['dev']])
rows = []
for which in ('whole', 'first third', 'middle third', 'last third'):
    vec = TfidfVectorizer(ngram_range=(1,2), min_df=3, max_features=100000, sublinear_tf=True)
    Xtr = vec.fit_transform([part(d['text'], which) for d in TR]); Xdv = vec.transform([part(d['text'], which) for d in DATA['dev']])
    for C_ in (0.3, 1, 3):
        m = LogisticRegression(C=C_, max_iter=400, class_weight='balanced').fit(Xtr, ytr)
        p = m.predict_proba(Xdv)[:, 1]; s = J.macro_scores(ydv, (p >= 0.5).astype(int))
        rows.append({'part': which, 'C': C_, 'macro_F1': round(s['macro_f1'],4), 'P': round(s['macro_precision'],3), 'R': round(s['macro_recall'],3), 'acc': round(s['accuracy'],3)})
res = pd.DataFrame(rows); display(res.sort_values('macro_F1', ascending=False))
print('best per part:'); print(res.groupby('part').macro_F1.max().round(4))
print('majority-class macro-F1 on dev:', round(J.macro_scores(ydv, np.full_like(ydv, int(ytr.mean() >= .5)))['macro_f1'], 4))